# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint


## Learning Objectives


At the end of the mini-hackathon you will be able to:
* Perform Data preprocessing
* Apply different ML algorithms on the **Titanic** dataset
* Perform VotingClassifier


## Dataset Description

The sinking of the Titanic is one of the most infamous shipwrecks in history.

On April 15, 1912, during her maiden voyage, the widely considered “unsinkable” RMS Titanic sank after colliding with an iceberg. Unfortunately, there weren’t enough lifeboats for everyone onboard, resulting in the death of many passengers and crew.

While there was some element of luck involved in surviving, it seems some groups of people were more likely to survive than others.

[ Data Set Link: Kaggle competition](https://www.kaggle.com/competitions/titanic)

<br/>

### Data Set Characteristics:

**PassengerId:** Id of the Passenger

**Survived:** Survived or Not information

**Pclass:** Socio-economic status (SES)
  * 1st = Upper
  * 2nd = Middle
  * 3rd = Lower

**Name:** Surname, First Names of the Passenger

**Sex:** Gender of the Passenger

**Age:** Age of the Passenger

**SibSp:**	No. of siblings/spouse of the passenger aboard the Titanic

**Parch:**	No. of parents/children of the passenger aboard the Titanic

**Ticket:**	Ticket number

**Fare:** Passenger fare

**Cabin:**	Cabin number

**Embarked:** Port of Embarkation
  * S = Southampton
  * C = Cherbourg
  * Q = Queenstown


## Problem Statement

Build a predictive model that answers the question: “what sort of people were more likely to survive?” using titanic's passenger data (ie name, age, gender, socio-economic class, etc).

In [ ]:
# @title Download the datasets
from IPython import get_ipython

ipython = get_ipython()

notebook="U1_MH1_Data_Munging" #name of the notebook

def setup():
    from IPython.display import HTML, display
    ipython.magic("sx wget https://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/titanic.csv")
    ipython.magic("sx wget https://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/test_titanic.csv")
    print("Data downloaded successfully")
    return

setup()

In [ ]:
!ls

## Exercise 1 - Load and Explore the Data (2 Marks)

* Understand different features in the training dataset
* Understand the data types of each column
* Notice the columns of missing values




#### Import Required Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression,Perceptron, SGDClassifier,RidgeClassifier,PassiveAggressiveClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier,BaggingClassifier

from sklearn.metrics import accuracy_score,classification_report

In [ ]:
# Load the dataset
# Load the dataset
df=pd.read_csv("titanic.csv")

In [ ]:
# Getting information about the dataset
df.dtypes

In [ ]:
df.info()

## Exercise 02: Split the data into train and test sets (1 Mark)
Note: Apply all your data preprocessing steps in the train set first and keep the test set aside.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2,random_state=42)

In [ ]:
df_train=pd.DataFrame(X_train)

In [ ]:
df_train.head()

## Exercise 03: Data Cleaning and Processing (15 Marks)
### 3.1 Working on the "Cabin" column (2 Marks)
Find unique entries in the Cabin column. We can label all passengers in two categories having a cabin or not. Check the data type(use: type) of each entry of the Cabin. Convert a string data type into '1' i.e. passengers with cabin and others into '0' i.e. passengers without cabin.  Write a function for the above operation and apply it to the cabin column and create another column with the name " Has_cabin" containing only 0 or 1 entries.





In [ ]:
display(df_train['Cabin'].unique())

In [ ]:
def has_cabin(cabin):
    return 1 if pd.notnull(cabin) else 0

In [ ]:
df_train['Has_cabin'] = df_train['Cabin'].apply(has_cabin)
display(df_train[['Cabin', 'Has_cabin']].head())

 ### 3.2 Working on "SibSp" & "Parch" columns (1 Mark)
Combine columns "SibSp" & "Parch" and create another column that represents the total passengers in one ticket with the name "family_size". In each ticket, there might be Siblings/Spouses (SibSp =Number of Siblings/Spouses Aboard) or Parents/Children (Parch=Number of Parents/Children Aboard ) along with the passenger who booked the ticket.

  

In [ ]:
df_train['family_size'] = df_train['SibSp'] + df_train['Parch']+1

In [ ]:
df_train['family_size'].head()

### 3.3 Working on the"Embarked" column (2 Marks)
The "embarked" column represents the port of Embarkation: Cherbourg(C), Queenstown(Q), and  Southampton(S ). Thus, the entries are of three categories in this column. Fill in the missing rows in this column. We can fill it with the most frequent category. Map these categorical string entries into numerical.



In [ ]:
# Fill missing Embarked values with most frequent
most_frequent_embarked = df_train['Embarked'].value_counts().idxmax()
df_train['Embarked'].fillna(most_frequent_embarked, inplace=True)

In [ ]:
df_train['Embarked'].head()

In [ ]:
# Map categorical to numerical
#embarked_mapping = {'S': 0, 'C': 1, 'Q': 2}
#df_train['Embarked'] = df_train['Embarked'].map(embarked_mapping)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

def embarck_one_hot_encoder(df1):
    # Create an instance of OneHotEncoder
  one_hot_encoder = OneHotEncoder(sparse_output=False) # Ensure dense array output
  embarked_encoded = one_hot_encoder.fit_transform(df1[['Embarked']])

  # Fix: Use df1.index instead of df_train.index to match the shape of embarked_encoded
  embarked_df = pd.DataFrame(embarked_encoded, columns=one_hot_encoder.get_feature_names_out(['Embarked']), index=df1.index)

  df1 = pd.concat([df1, embarked_df], axis=1)
  df1.drop('Embarked', axis=1, inplace=True)
  return df1

In [ ]:
df_train=embarck_one_hot_encoder(df_train)
print(df_train.columns)

In [ ]:
df_train.head()

### 3.4 Working on the "Age" column (2 Marks)
find the number of NaN entries in the age column and their row index. Calculate the mean, Standard deviation of the Age column and check the distribution of the age column.We can fill the missing values with randomly generated integer values between (mean+Standard deviation, mean-Standard deviation). Use : np.isnan; np.random.randint; concept of slicing dataframe. Convert the age column as an integer data type.



In [ ]:
df_train["Age"].isna().sum()

In [ ]:
def impute_age(df1):
  np.random.seed(42) #ensure same random Age Values every run
  index_anull=df1[df1['Age'].isna()].index
  ms_sum=np.mean(df1['Age'])+np.std(df1['Age'])
  ms_sub=np.mean(df1['Age'])-np.std(df1['Age'])
  for i in index_anull:
    df1.loc[i,'Age']=np.random.randint(ms_sub,ms_sum)
  df1['Age']=df1['Age'].astype(int)
  return df1


In [ ]:
df_train=impute_age(df_train)
df_train['Age'].isna().sum()

### 3.5 Working on "sex" column (1 Mark)
Map the Sex column as 'female' : 0, 'male': 1, and convert it into an integer data type.



In [ ]:
df_train['Sex'].value_counts()

In [ ]:
df_train['Sex']=df_train['Sex'].map({'male':1,'female':0}).astype(int)

In [ ]:
df_train['Sex']

### 3.6  Optional- Working on the "Name" column :
Fetch titles from the name. We can map these titles with numbers and convert them into an integer. Use: concept of the regular expression.

### 3.7 Optional- Working on the "Fare" column :
We can convert face into categorical entries like Low, Medium, and High.



In [ ]:
import regex as re
def get_title(name):
    search = re.search(' ([A-Za-z]+)\.', name)

    if search:
        return search.group(1)

    return ""

In [ ]:
df_train['Title'] = df_train['Name'].apply(get_title)

In [ ]:
le_title_encoder=LabelEncoder()
df_train['Title']=le_title_encoder.fit_transform(df_train['Title'])

In [ ]:
df_train['Title'].head()

### 3.8 Drop the columns (1 Mark)

Drop the columns: - "PassengerId", "Name",  "SibSp" & "Parch", "Tickets", "Cabin"



In [ ]:
df_train.drop(columns=['PassengerId', 'Name', 'SibSp', 'Parch', 'Ticket', 'Cabin'],inplace=True)

In [ ]:
df_train.columns

In [ ]:
df["FamilySize"] = (
    df["SibSp"] +
    df["Parch"] +
    1
)

In [ ]:
sns.countplot(
    x='FamilySize',
    hue='Survived',
    data=df
)

In [ ]:
df["IsAlone"] = (
    df["FamilySize"] == 1
).astype(int)

In [ ]:
sns.countplot(
    x='IsAlone',
    hue='Survived',
    data=df
)

In [ ]:
df_train['is_alone']=df_train['family_size'].apply(lambda x: 1 if x == 1 else 0).astype(int)

In [ ]:
df_train['is_alone'].head()

In [ ]:
df_train['is_child']=df_train['Age'].apply(lambda x: 1 if x <=12 else 0).astype(int)

### 3.9 Apply Standard Scalar (1 Mark)

In [ ]:
# Fit scaler on training data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_train_scaled = scaler.fit_transform(df_train)

# Convert to DataFrame
df_train_scaled = pd.DataFrame(df_train_scaled, columns=df_train.columns)


In [ ]:
df_train_scaled.head()

In [ ]:
df_train_scaled.corr()

### 3.10 Create a single function for preprocessing the test set (X_test) and apply it. (4 Marks)
#### **Note**: All the pre-processing steps that were applied on the train set before ML Modelling are also applied on the test set before passing through the predict function.

In [ ]:
## Create a function
def preprocess(df_input):
  # Create a copy to avoid SettingWithCopyWarning and ensure modifications are made on an independent DataFrame
  df_input = df_input.copy()

  df_input['Has_cabin'] = df_input['Cabin'].notna().astype(int)
  df_input['family_size'] = df_input['SibSp'] + df_input['Parch']+1

  # Check if Embarked is object type before mapping to avoid re-mapping already numerical data
  #if df_input['Embarked'].dtype == 'object':
  #    df_input['Embarked']=df_input['Embarked'].map({'S':1,'C':2,'Q':3}).fillna(1).astype(int)
  # If one-hot encoding for Embarked is intended for test data, uncomment and correctly implement the call here
  most_frequent_embarked = df_input['Embarked'].value_counts().idxmax()
  df_input['Embarked'].fillna(most_frequent_embarked, inplace=True)
  df_input = embarck_one_hot_encoder(df_input)

  df_input=impute_age(df_input)
  #df_input['is_child']=df_input['Age'].apply(lambda x: 1 if x <=12 else 0).astype(int)

  # Check if Sex is object type before mapping to avoid re-mapping already numerical data
  if df_input['Sex'].dtype == 'object':
      df_input['Sex']=df_input['Sex'].map({'male':1,'female':0}).astype(int)

  df_input['Title'] = df_input['Name'].apply(get_title)
  # Handle unseen labels in 'Title' before transformation
  # Use .loc with .isin() for robust replacement of unseen titles
  unseen_titles_mask = ~df_input['Title'].isin(le_title_encoder.classes_)
  df_input.loc[unseen_titles_mask, 'Title'] = 'Mr'

  df_input['Title']=le_title_encoder.transform(df_input['Title'])

  df_input['is_alone']=df_input['family_size'].apply(lambda x: 1 if x == 1 else 0).astype(int)
  df_input['is_child']=df_input['Age'].apply(lambda x: 1 if x <=12 else 0).astype(int)

  df_input.drop(columns=['PassengerId', 'Name', 'SibSp', 'Parch', 'Ticket', 'Cabin'],inplace=True,errors='ignore')
  return df_input

In [ ]:
## Applyting above function
df_test=pd.DataFrame(X_test)
df_test_preprocess=preprocess(df_test)


In [ ]:
df_test_preprocess.head()

### 3.11 Apply standard Scalar transformation to x_test (1 Mark)

In [ ]:
#expected_columns_order = ['Pclass', 'Sex', 'Age', 'Fare', 'Has_cabin', 'family_size', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Title']

#df_test_preprocess_reordered = df_test_preprocess[expected_columns_order]

X_test_scaled=scaler.transform(df_test_preprocess)
X_test_scaled.shape

In [ ]:
# Convert X_test_scaled to DataFrame with correct column names
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=df_train_scaled.columns)

In [ ]:
X_test_scaled_df.head()

## Exercise  4. Apply Multiple ML Algo. along with  Ensemble Technique (Voting classifier) and display the accuracy (7 Marks)
#### Expected Accuracy >= 80%  


In [ ]:
models={
    'LogisticRegression':LogisticRegression(C=100,max_iter=5000, random_state=42),
    'KNeighborsClassifier':KNeighborsClassifier(n_neighbors=5),
    'SVC':SVC(probability=True,C=100),
    'Perceptron':Perceptron(),
    'SGDClassifier':SGDClassifier(loss='log_loss', max_iter=5000, random_state=42),
    'VotingClassifier':VotingClassifier(estimators=[#('lr', LogisticRegression()),
                                                    ('knn', KNeighborsClassifier(n_neighbors=5)),
                                                    ('bg',BaggingClassifier(n_estimators=500,random_state=42)),
                                                     ('svc',SVC(kernel='rbf',probability=True,C=100))], voting='soft'),
    'BaggingClassifier':BaggingClassifier(n_estimators=300, random_state=42)
}

In [ ]:
scores = {}
for name,model in models.items():
  model.fit(df_train_scaled,y_train)
  score=model.score(X_test_scaled_df,y_test)
  scores[name] = score

for name, score in scores.items():
    print(f"{name:15s}: {score:f}{'>80 %' if score>=0.8 else ''}")

In [ ]:
!pip install optuna

In [ ]:
import optuna
from sklearn.metrics import accuracy_score

# Assuming df_train_scaled, y_train, X_test_scaled_df, y_test are already defined and scaled

def objective(trial):
    classifier_name = trial.suggest_categorical('classifier', ['LogisticRegression', 'KNeighborsClassifier', 'SVC', 'BaggingClassifier'])

    if classifier_name == 'LogisticRegression':
        log_reg_c = trial.suggest_float('log_reg_C', 1e-3, 1e3, log=True)
        model = LogisticRegression(C=log_reg_c, solver='liblinear', random_state=42)
    elif classifier_name == 'KNeighborsClassifier':
        n_neighbors = trial.suggest_int('knn_n_neighbors', 3, 20)
        model = KNeighborsClassifier(n_neighbors=n_neighbors)
    elif classifier_name == 'SVC':
        svc_c = trial.suggest_float('svc_C', 1e-3, 1e3, log=True)
        svc_gamma = trial.suggest_categorical('svc_gamma', ['scale', 'auto'])
        model = SVC(C=svc_c, gamma=svc_gamma, probability=True, random_state=42)
    elif classifier_name == 'BaggingClassifier':
        n_estimators = trial.suggest_int('bagging_n_estimators', 10, 200)
        model = BaggingClassifier(n_estimators=n_estimators, random_state=42)

    model.fit(df_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled_df)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Create an Optuna study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50) # Run 50 trials

print("Number of finished trials: ", len(study.trials))
print("Best trial:")
trial = study.best_trial

print(f"  Value: {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

# You can also get the best model directly
# best_model_params = trial.params
# if best_model_params['classifier'] == 'LogisticRegression':
#     best_model = LogisticRegression(C=best_model_params['log_reg_C'], solver='liblinear', random_state=42)
# elif best_model_params['classifier'] == 'KNeighborsClassifier':
#     best_model = KNeighborsClassifier(n_neighbors=best_model_params['knn_n_neighbors'])
# elif best_model_params['classifier'] == 'SVC':
#     best_model = SVC(C=best_model_params['svc_C'], gamma=best_model_params['svc_gamma'], probability=True, random_state=42)
# elif best_model_params['classifier'] == 'BaggingClassifier':
#     best_model = BaggingClassifier(n_estimators=best_model_params['bagging_n_estimators'], random_state=42)
# best_model.fit(df_train_scaled, y_train)
# print("Best model accuracy on test set:", best_model.score(X_test_scaled_df, y_test))


## Exercise  5. Pre-process the test_set (3 Marks)
Again we have to apply the same preprocess function and standard scaler on this test set before passing through predict function.

#### Understanding the test set:

In [ ]:
# Load the dataset
df_test_set = pd.read_csv("test_titanic.csv")

In [ ]:
df_test_set.head()

#### Note: In the initial train set there were no missing entries in the "Fare" column. But, now for the submission test set, there is one missing entry in this column.

#### There will be a minor change in the preprocess function to address the above issue.

In [ ]:
df_test_set.info()


In [ ]:
## Create a function
def preprocess_v2(df_input):
  # Create a copy to avoid SettingWithCopyWarning and ensure modifications are made on an independent DataFrame
  df_input = df_input.copy()

  df_input['Has_cabin'] = df_input['Cabin'].notna().astype(int)
  df_input['family_size'] = df_input['SibSp'] + df_input['Parch']+1
  #df_input['is_alone']=df_input['family_size'].apply(lambda x: 1 if x == 1 else 0).astype(int)

  # Check if Embarked is object type before mapping to avoid re-mapping already numerical data
  #if df_input['Embarked'].dtype == 'object':
  #    df_input['Embarked']=df_input['Embarked'].map({'S':1,'C':2,'Q':3}).fillna(1).astype(int)
  # If one-hot encoding for Embarked is intended for test data, uncomment and correctly implement the call here
  most_frequent_embarked = df_input['Embarked'].value_counts().idxmax()
  df_input['Embarked'].fillna(most_frequent_embarked, inplace=True)
  df_input = embarck_one_hot_encoder(df_input)

  df_input=impute_age(df_input)
  #df_input['is_child']=df_input['Age'].apply(lambda x: 1 if x <=12 else 0).astype(int)

  # Check if Sex is object type before mapping to avoid re-mapping already numerical data
  if df_input['Sex'].dtype == 'object':
      df_input['Sex']=df_input['Sex'].map({'male':1,'female':0}).astype(int)

  df_input['Title'] = df_input['Name'].apply(get_title)
  # Handle unseen labels in 'Title' before transformation
  # Use .loc with .isin() for robust replacement of unseen titles
  unseen_titles_mask = ~df_input['Title'].isin(le_title_encoder.classes_)
  df_input.loc[unseen_titles_mask, 'Title'] = 'Mr'

  df_input['Title']=le_title_encoder.transform(df_input['Title'])

  df_input.drop(columns=['PassengerId', 'Name', 'SibSp', 'Parch', 'Ticket', 'Cabin'],inplace=True,errors='ignore')

  #Handle missing fare values
  df_input['Fare'].fillna(df_input['Fare'].mean(), inplace=True)

  df_input['is_alone']=df_input['family_size'].apply(lambda x: 1 if x == 1 else 0).astype(int)
  df_input['is_child']=df_input['Age'].apply(lambda x: 1 if x <=12 else 0).astype(int)
  return df_input

In [ ]:
df_test_set_preprocess=preprocess_v2(df_test_set)

In [ ]:
df_test_set_preprocess.head()


In [ ]:
#Scaling function
#expected_columns_order = ['Pclass', 'Sex', 'Age', 'Fare', 'Has_cabin', 'family_size', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Title']

#df_test_preprocess_reordered = df_test_preprocess[expected_columns_order]

X_test_scaled=scaler.transform(df_test_set_preprocess)
X_test_scaled.shape

## Exercise  6. Prediction for test data (2 Mark)

In [ ]:
# Get PassengerId from original test file (dropped during preprocessing)
passenger_ids = pd.read_csv("test_titanic.csv")['PassengerId']

# Use best model from Exercise 4 - VotingClassifier (84.9% accuracy)
best_model = models['VotingClassifier']

# Predict on the scaled test set
y_pred_test = best_model.predict(df_test_set_preprocess)
print
# Create submission DataFrame
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
     'Survived': y_pred_test
 })

print(submission.head(10))
# print("\nTotal predictions:", len(submission))
# print("Survived distribution:\n", submission['Survived'].value_counts())

# # Save to CSV
# submission.to_csv('titanic_submission.csv', index=False)
# print("\nSaved to titanic_submission.csv")
